# 有馬記念用のDataをNetkeibaから収集するためのコード

### まず、過去3年分くらいのレース情報を取得したいが、GIに限ることにする

In [1]:
import pandas as pd
import numpy as np
import datetime
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
#import lightgbm as lgb
import requests
from bs4 import BeautifulSoup
import time
import re
from urllib.request import Request, urlopen
#import optuna.integration.lightgbm as lgb_o
from itertools import combinations, permutations
import matplotlib.pyplot as plt


## ウェブスクレイピングで情報収集

In [2]:
import random

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Edg/115.0.0.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
]

random.choice(USER_AGENTS)

'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55'

In [33]:
class Results:
    @staticmethod
    def scrape(race_id_list):
        """
        レース結果データをスクレイピングする関数
        Parameters:
        ----------
        race_id_list : list
            レースIDのリスト
        Returns:
        ----------
        race_results_df : pandas.DataFrame
            全レース結果データをまとめてDataFrame型にしたもの
        """
        #race_idをkeyにしてDataFrame型を格納
        race_results = {}
        for race_id in tqdm(race_id_list):
            time.sleep(1)
            try:
                url = "https://db.netkeiba.com/race/" + race_id
                headers = {'User-Agent': random.choice(USER_AGENTS)}
                html = requests.get(url, headers=headers)
                html.encoding = "EUC-JP"
                # メインとなるテーブルデータを取得
                df = pd.read_html(html.text)[0]
                # 列名に半角スペースがあれば除去する
                df = df.rename(columns=lambda x: x.replace(' ', ''))
                # 天候、レースの種類、コースの長さ、馬場の状態、日付をスクレイピング
                soup = BeautifulSoup(html.text, "html.parser")
                texts = (
                    soup.find("div", attrs={"class": "data_intro"}).find_all("p")[0].text
                    + soup.find("div", attrs={"class": "data_intro"}).find_all("p")[1].text
                )
                info = re.findall(r'\w+', texts)
                for text in info:
                    if text in ["芝", "ダート"]:
                        df["race_type"] = [text] * len(df)
                    if "障" in text:
                        df["race_type"] = ["障害"] * len(df)
                    if "m" in text:
                        df["course_len"] = [int(re.findall(r"\d+", text)[-1])] * len(df)
                    if text in ["良", "稍重", "重", "不良"]:
                        df["ground_state"] = [text] * len(df)
                    if text in ["曇", "晴", "雨", "小雨", "小雪", "雪"]:
                        df["weather"] = [text] * len(df)
                    if "年" in text:
                        df["date"] = [text] * len(df)
                #馬ID、騎手IDをスクレイピング
                horse_id_list = []
                horse_a_list = soup.find("table", attrs={"summary": "レース結果"}).find_all(
                    "a", attrs={"href": re.compile("^/horse")}
                )
                for a in horse_a_list:
                    horse_id = re.findall(r"\d+", a["href"])
                    horse_id_list.append(horse_id[0])
                jockey_id_list = []
                jockey_a_list = soup.find("table", attrs={"summary": "レース結果"}).find_all(
                    "a", attrs={"href": re.compile("^/jockey")}
                )
                for a in jockey_a_list:
                    jockey_id = re.findall(r"\d+", a["href"])
                    jockey_id_list.append(jockey_id[0])
                df["horse_id"] = horse_id_list
                df["jockey_id"] = jockey_id_list
                #インデックスをrace_idにする
                df.index = [race_id] * len(df)
                race_results[race_id] = df
            #存在しないrace_idを飛ばす
            except IndexError:
                continue
            except AttributeError: #存在しないrace_idでAttributeErrorになるページもあるので追加
                continue
            #wifiの接続が切れた時などでも途中までのデータを返せるようにする
            except Exception as e:
                print(e)
                break
            #Jupyterで停止ボタンを押した時の対処
            except:
                break
        #pd.DataFrame型にして一つのデータにまとめる
        race_results_df = pd.concat([race_results[key] for key in race_results])
        return race_results_df

In [34]:
# race_id_list = []
# for place in range(1, 11, 1):
#     for kai in range(1, 7, 1):
#         for day in range(1, 13, 1):
#             for r in range(1, 13, 1):
#                 race_id = "2025" + str(place).zfill(2) + str(kai).zfill(2) + str(day).zfill(2) + str(r).zfill(2)
#                 race_id_list.append(race_id)

# results = Results.scrape(race_id_list)

In [30]:
import pandas as pd
import requests
import random

# ユーザーエージェントのリスト（適宜追加してください）
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
]

# 取得したいURLのリスト
urls = [
    # 1ページ目 (page指定なし)
    'https://db.netkeiba.com/?pid=race_list&word=&track%5B%5D=1&start_year=2025&start_mon=1&end_year=2025&end_mon=12&jyo%5B%5D=01&jyo%5B%5D=02&jyo%5B%5D=03&jyo%5B%5D=04&jyo%5B%5D=05&jyo%5B%5D=06&jyo%5B%5D=07&jyo%5B%5D=08&jyo%5B%5D=09&jyo%5B%5D=10&jyo%5B%5D=30&jyo%5B%5D=35&jyo%5B%5D=36&jyo%5B%5D=42&jyo%5B%5D=43&jyo%5B%5D=44&jyo%5B%5D=45&jyo%5B%5D=46&jyo%5B%5D=47&jyo%5B%5D=48&jyo%5B%5D=50&jyo%5B%5D=51&jyo%5B%5D=54&jyo%5B%5D=55&jyo%5B%5D=65&grade%5B%5D=1&grade%5B%5D=2&grade%5B%5D=3&kyori_min=&kyori_max=&sort=date&list=100',
    # 2ページ目 (page=2)
    "https://db.netkeiba.com//?pid=race_list&word=&track%5B0%5D=1&start_year=2025&start_mon=1&end_year=2025&end_mon=12&jyo%5B0%5D=01&jyo%5B1%5D=02&jyo%5B2%5D=03&jyo%5B3%5D=04&jyo%5B4%5D=05&jyo%5B5%5D=06&jyo%5B6%5D=07&jyo%5B7%5D=08&jyo%5B8%5D=09&jyo%5B9%5D=10&jyo%5B10%5D=30&jyo%5B11%5D=35&jyo%5B12%5D=36&jyo%5B13%5D=42&jyo%5B14%5D=43&jyo%5B15%5D=44&jyo%5B16%5D=45&jyo%5B17%5D=46&jyo%5B18%5D=47&jyo%5B19%5D=48&jyo%5B20%5D=50&jyo%5B21%5D=51&jyo%5B22%5D=54&jyo%5B23%5D=55&jyo%5B24%5D=65&grade%5B0%5D=1&grade%5B1%5D=2&grade%5B2%5D=3&kyori_min=&kyori_max=&sort=date&list=100&page=2"
]

# データフレームを格納するリスト
df_list = []

for url in urls:
    try:
        headers = {'User-Agent': random.choice(USER_AGENTS)}
        
        # データの取得
        response = requests.get(url, headers=headers)
        response.encoding = "EUC-JP" # 文字化け対策
        
        # テーブルデータの読み込み
        # pd.read_htmlはリストを返すため、[0]で最初のテーブルを取得
        dfs = pd.read_html(response.text)
        
        if len(dfs) > 0:
            df_temp = dfs[0]
            df_list.append(df_temp)
            print(f"取得成功: {url[:60]}... (行数: {len(df_temp)})")
        else:
            print(f"テーブルが見つかりませんでした: {url}")
            
    except Exception as e:
        print(f"エラーが発生しました: {e} URL: {url}")

# 結合処理
if df_list:
    # 複数のDataFrameを縦に結合 (ignore_index=Trueでindexを振り直す)
    final_df = pd.concat(df_list, ignore_index=True)
    
    # 列名の半角スペース除去
    final_df = final_df.rename(columns=lambda x: x.replace(' ', ''))
    
    print("-" * 30)
    print(f"統合完了。合計行数: {len(final_df)}")
    print(final_df.head()) # 確認用表示
else:
    print("データが取得できませんでした。")

/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_94046/1425024747.py:32: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(response.text)


取得成功: https://db.netkeiba.com/?pid=race_list&word=&track%5B%5D=1&s... (行数: 100)
取得成功: https://db.netkeiba.com//?pid=race_list&word=&track%5B0%5D=1... (行数: 10)
------------------------------
統合完了。合計行数: 110
          開催日    開催 天気   R           レース名  映像     距離  頭数 馬場     タイム  \
0  2025/12/14  5阪神4  晴  11  阪神ジュベナイルF(GI) NaN  芝1600  18  良  1:32.6   
1  2025/12/13  5中京3  晴  11    中日新聞杯(GIII) NaN  芝2000  18  良  1:57.6   
2  2025/12/06  5中山1  晴  11   ステイヤーズS(GII) NaN  芝3600  14  良  3:47.2   
3  2025/12/06  5阪神1  晴  11     鳴尾記念(GIII) NaN  芝1800  14  良  1:43.7   
4  2025/11/30  5東京8  晴  12      ジャパンC(GI) NaN  芝2400  17  良  2:20.3   

         ペース        勝ち馬    騎手      調教師        2着馬      3着馬  
0  33.7-35.3     スターアニス  松山弘平  [西]高野友和     ギャラボーグ  タイセイボーグ  
1  35.1-34.1  シェイクユアハート  古川吉洋    [西]宮徹   レッドバリエンテ  ジューンテイク  
2  39.2-34.3     ホーエリート  戸崎圭太  [東]田島俊明  マイネルカンパーナ   クロミナンス  
3  33.9-34.7   デビットバローズ  岩田望来  [西]上村洋行    センツブラッド  マテンロウレオ  
4  34.5-34.6     カランダガン  バルザロ  [外]グラファ  マスカレードボール  ダノンデサイル  


/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_94046/1425024747.py:32: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(response.text)


In [31]:
final_df

,開催日,開催,天気,R,レース名,映像,距離,頭数,馬場,タイム,ペース,勝ち馬,騎手,調教師,2着馬,3着馬
0,2025/12/14,5阪神4,晴,11,阪神ジュベナイルF(GI),NaN,芝1600,18,良,1:32.6,33.7-35.3,スターアニス,松山弘平,[西]高野友和,ギャラボーグ,タイセイボーグ
1,2025/12/13,5中京3,晴,11,中日新聞杯(GIII),NaN,芝2000,18,良,1:57.6,35.1-34.1,シェイクユアハート,古川吉洋,[西]宮徹,レッドバリエンテ,ジューンテイク
2,2025/12/06,5中山1,晴,11,ステイヤーズS(GII),NaN,芝3600,14,良,3:47.2,39.2-34.3,ホーエリート,戸崎圭太,[東]田島俊明,マイネルカンパーナ,クロミナンス
3,2025/12/06,5阪神1,晴,11,鳴尾記念(GIII),NaN,芝1800,14,良,1:43.7,33.9-34.7,デビットバローズ,岩田望来,[西]上村洋行,センツブラッド,マテンロウレオ
4,2025/11/30,5東京8,晴,12,ジャパンC(GI),NaN,芝2400,17,良,2:20.3,34.5-34.6,カランダガン,バルザロ,[外]グラファ,マスカレードボール,ダノンデサイル
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,2025/01/19,1中京7,晴,11,日経新春杯(GII),NaN,芝2200,16,良,2:09.8,34.5-37.4,ロードデルレイ,西村淳也,[西]中内田充,ショウナンラプンタ,マイネルエンペラー
106,2025/01/13,1中京5,晴,11,日刊スポシンザン記念(GIII),NaN,芝1600,15,良,1:34.6,35.1-35.8,リラエンブレム,浜中俊,[西]武幸四郎,アルテヴェローチェ,ウォーターガーベラ
107,2025/01/12,1中山4,曇,11,フェアリーS(GIII),NaN,芝1600,16,良,1:32.8,34.1-35.5,エリカエクスプレス,戸崎圭太,[西]杉山晴紀,ティラトーレ,エストゥペンダ
108,2025/01/05,1中山1,曇,11,日刊スポ賞中山金杯(GIII),NaN,芝2000,18,良,1:58.1,34.9-35.7,アルナシーム,藤岡佑介,[西]橋口慎介,マイネルモーント,ボーンディスウェイ


In [32]:
import pandas as pd
import re

# 1. 場所名とネット競馬ID(2桁)の対応辞書
# ユーザー様のリスト順序等を考慮し、ネット競馬で実際に使われるコードを定義しています
place_map = {
    # JRA (01-10)
    '札幌': '01', '函館': '02', '福島': '03', '新潟': '04', '東京': '05', 
    '中山': '06', '中京': '07', '京都': '08', '阪神': '09', '小倉': '10',
    # NAR・地方 (30-65) ※主な場所のID
    '門別': '30', '盛岡': '35', '水沢': '36', 
    '浦和': '42', '船橋': '43', '大井': '44', '川崎': '45', 
    '金沢': '46', '笠松': '47', '名古屋': '48', 
    '園田': '50', '姫路': '51', 
    '高知': '54', '佐賀': '55', 
    '帯広(ば)': '65', '帯広': '65' # 表記揺れ対応
}

def generate_race_id(row):
    """
    行ごとのデータからRaceID(2025xxxxxxxx)を生成する関数
    想定入力: 開催='5阪神4' または '5回阪神4日', R=11
    """
    try:
        # 開催情報の文字列取得
        kaisai_str = str(row['開催'])
        
        # 正規表現で「数字」「場所名」「数字」を抽出
        # 例: "5阪神4" -> group1=5, group2=阪神, group3=4
        # "回"や"日"が含まれていても無視して数字と場所だけ抜くように調整
        match = re.search(r'(\d+)[回]*(\D+?)(\d+)[日]*', kaisai_str)
        
        if match:
            kai = match.group(1).zfill(2)   # 回 (5 -> 05)
            place_name = match.group(2)     # 場所名 (阪神)
            day = match.group(3).zfill(2)   # 日 (4 -> 04)
            
            # 場所名からIDを取得
            place_id = place_map.get(place_name)
            
            # レース番号 (11 -> 11)
            race_num = str(row['R']).zfill(2)
            
            if place_id:
                # 2025 + 場所ID + 回 + 日 + レース番号
                # 例: 2025 + 09 + 05 + 04 + 11
                return "2025" + place_id + kai + day + race_num
            else:
                return None # 対応する場所名がない場合
        else:
            return None # マッチしない場合
            
    except Exception as e:
        return None

# 2. 関数を適用して新しい列 'race_id' を作成
final_df['race_id'] = final_df.apply(generate_race_id, axis=1)

# 確認用: 変換できたデータと元の列を表示
print(final_df[['開催', 'R', 'race_id']].head(10))

# race_idが作成できたものだけを抽出したい場合
# final_df = final_df.dropna(subset=['race_id'])

     開催   R       race_id
0  5阪神4  11  202509050411
1  5中京3  11  202507050311
2  5中山1  11  202506050111
3  5阪神1  11  202509050111
4  5東京8  12  202505050812
5  4京都8  12  202508040812
6  4京都7  11  202508040711
7  5東京6  11  202505050611
8  4京都6  11  202508040611
9  3福島5  11  202503030511


In [35]:
race_id_list = final_df['race_id']
results = Results.scrape(race_id_list)

  0%|          | 0/110 [00:00<?, ?it/s]

/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_94046/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html.text)[0]
/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_94046/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html.text)[0]
/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_94046/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html.text)[0]
/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_94046/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecate

In [37]:
results.to_pickle('results.pickle')
#results = pd.read_pickle('results.pickle')

In [38]:
results

,着順,枠番,馬番,馬名,性齢,斤量,騎手,タイム,着差,単勝,人気,馬体重,調教師,course_len,weather,race_type,ground_state,date,horse_id,jockey_id
202506050111,1,3,3,ホーエリート,牝4,55.0,戸崎圭太,3:47.2,NaN,4.4,2.0,476(-6),[東] 田島俊明,3600,晴,芝,良,2025年12月06日,2021104855,05386
202506050111,2,3,4,マイネルカンパーナ,牡5,57.0,津村明秀,3:47.3,3/4,11.6,5.0,414(+2),[東] 青木孝文,3600,晴,芝,良,2025年12月06日,2020105798,01092
202506050111,3,5,7,クロミナンス,牡8,57.0,ルメール,3:47.3,ハナ,3.5,1.0,494(0),[東] 尾関知人,3600,晴,芝,良,2025年12月06日,2017105074,05339
202506050111,4,5,8,ブレイヴロッカー,セ5,57.0,荻野極,3:47.3,クビ,18.3,9.0,456(+2),[西] 本田優,3600,晴,芝,良,2025年12月06日,2020102870,01160
202506050111,5,7,12,ワープスピード,牡6,57.0,菅原明良,3:47.5,3/4,18.0,8.0,504(+6),[東] 高木登,3600,晴,芝,良,2025年12月06日,2019105312,01179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202507010111,12,4,8,オーキッドロマンス,牡4,57.0,丸田恭介,1:35.1,1.1/4,79.4,13.0,490(0),[東] 手塚貴久,1600,晴,芝,良,2025年01月05日,2021103323,01117
202507010111,13,5,10,セルバーグ,牡6,57.0,田口貫太,1:35.3,1.1/2,18.6,9.0,444(+12),[西] 鈴木孝志,1600,晴,芝,良,2025年01月05日,2019106670,01208
202507010111,14,6,12,フィールシンパシー,牝6,54.0,坂井瑠星,1:35.7,2.1/2,12.8,7.0,456(-2),[東] 小島茂之,1600,晴,芝,良,2025年01月05日,2019105768,01163
202507010111,15,2,4,ゴールデンシロップ,牡7,55.0,原優介,1:35.7,ハナ,63.3,12.0,544(+4),[東] 鈴木慎太,1600,晴,芝,良,2025年01月05日,2018110043,01184


In [39]:
#馬の過去成績データを処理するクラス
class HorseResults:
    @staticmethod
    def scrape(horse_id_list):
        """
        馬の過去成績データをスクレイピングする関数

        Parameters:
        ----------
        horse_id_list : list
            馬IDのリスト

        Returns:
        ----------
        horse_results_df : pandas.DataFrame
            全馬の過去成績データをまとめてDataFrame型にしたもの
        """

        #horse_idをkeyにしてDataFrame型を格納
        horse_results = {}
        for horse_id in tqdm(horse_id_list):
            time.sleep(1)
            try:
                url = 'https://db.netkeiba.com/horse/' + horse_id
                headers = {'User-Agent': random.choice(USER_AGENTS)}
                html = requests.get(url, headers=headers)
                html.encoding = "EUC-JP"
                df = pd.read_html(html.text)[2]
                df.index = [horse_id] * len(df)
                horse_results[horse_id] = df
            except IndexError:
                continue
            except Exception as e:
                print(e)
                break
            except:
                break

        #pd.DataFrame型にして一つのデータにまとめる        
        horse_results_df = pd.concat([horse_results[key] for key in horse_results])

        return horse_results_df

In [42]:
horse_id_list = results['horse_id'].unique()
horse_results = HorseResults.scrape(horse_id_list[0:2])
horse_results #jupyterで出力

  0%|          | 0/2 [00:00<?, ?it/s]

/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_94046/3171480521.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html.text)[2]
/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_94046/3171480521.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html.text)[2]


ValueError: No objects to concatenate